# Préparation de données pour le machine learning

Ce notebook présente plusieurs transformations courantes avant l'entraînement d'un modèle :

1. encodage déterministe de variables catégorielles ;
2. traitement des valeurs aberrantes avec la règle de l'IQR ;
3. classement à partir de données agrégées ;
4. tokenisation simple d'une phrase ;
5. standardisation ligne par ligne d'une matrice ;
6. encodage *one-hot* avec pandas et scikit-learn.

Les fonctions évitent de modifier les données sources et valident les colonnes attendues. Les jeux de données ci-dessous sont synthétiques et servent uniquement à rendre les exemples reproductibles.


In [21]:
from __future__ import annotations

import string
from collections.abc import Iterable

import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", None)


## 1. Encodage déterministe de catégories

Chaque valeur distincte reçoit un entier positif. Le tri alphabétique rend le résultat reproductible, tandis que les dictionnaires retournés permettent de décoder les valeurs si nécessaire.

> Cet encodage est une pseudonymisation, pas une anonymisation cryptographique : les correspondances ne doivent pas être publiées avec des données sensibles.


In [22]:
def encode_categories(
    dataframe: pd.DataFrame,
    columns: Iterable[str],
    *,
    start: int = 1,
) -> tuple[pd.DataFrame, dict[str, dict[str, dict]]]:
    """Encode des catégories triées et retourne les tables de correspondance."""
    columns = list(columns)
    missing = sorted(set(columns) - set(dataframe.columns))
    if missing:
        raise KeyError(f"Colonnes absentes : {missing}")

    encoded = dataframe.copy()
    mappings: dict[str, dict[str, dict]] = {}

    for column in columns:
        values = sorted(encoded[column].dropna().unique())
        to_code = {value: index for index, value in enumerate(values, start=start)}
        to_value = {code: value for value, code in to_code.items()}
        encoded[column] = encoded[column].map(to_code).astype("Int64")
        mappings[column] = {"to_code": to_code, "to_value": to_value}

    return encoded, mappings


orders = pd.DataFrame(
    {
        "CustomerName": [
            "John Smith", "Tom Smith", "Jane Doe",
            "Alex William", "John Smith", "Jane Doe",
        ],
        "Order": ["Laptop", "Router", "TV", "Laptop", "TV", "Cable"],
    }
)

encoded_orders, category_mappings = encode_categories(
    orders, columns=["CustomerName", "Order"]
)
encoded_orders


,CustomerName,Order
0,3,2
1,4,3
2,2,4
3,1,2
4,3,4
5,2,1


In [23]:
# Exemple de décodage d'une colonne.
decoded_customers = encoded_orders["CustomerName"].map(
    category_mappings["CustomerName"]["to_value"]
)
decoded_customers


0      John Smith
1       Tom Smith
2        Jane Doe
3    Alex William
4      John Smith
5        Jane Doe
Name: CustomerName, dtype: object

## 2. Traitement des valeurs aberrantes

La règle de Tukey considère comme aberrante une valeur située hors de l'intervalle
$[Q_1 - 1{,}5 \, IQR,\; Q_3 + 1{,}5 \, IQR]$.

La fonction remplace ces valeurs par la valeur observée non aberrante la plus proche. Elle retourne également un rapport exploitable dans un pipeline de contrôle qualité.


In [24]:
def cap_outliers_iqr(
    dataframe: pd.DataFrame,
    columns: Iterable[str] | None = None,
    *,
    factor: float = 1.5,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Écrête les valeurs aberrantes selon l'IQR sans modifier l'entrée."""
    if factor <= 0:
        raise ValueError("factor doit être strictement positif")

    cleaned = dataframe.copy()
    selected = list(columns) if columns is not None else list(
        cleaned.select_dtypes(include="number").columns
    )
    missing = sorted(set(selected) - set(cleaned.columns))
    if missing:
        raise KeyError(f"Colonnes absentes : {missing}")

    report = []
    for column in selected:
        series = cleaned[column]
        if not pd.api.types.is_numeric_dtype(series):
            raise TypeError(f"La colonne {column!r} doit être numérique")

        q1, q3 = series.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower_fence, upper_fence = q1 - factor * iqr, q3 + factor * iqr
        valid = series.between(lower_fence, upper_fence) | series.isna()
        non_outliers = series[valid].dropna()
        lower_cap = non_outliers.min() if not non_outliers.empty else np.nan
        upper_cap = non_outliers.max() if not non_outliers.empty else np.nan
        outlier_mask = ~valid

        cleaned[column] = series.clip(lower=lower_cap, upper=upper_cap)
        report.append(
            {
                "column": column,
                "outlier_count": int(outlier_mask.sum()),
                "lower_fence": lower_fence,
                "upper_fence": upper_fence,
                "lower_cap": lower_cap,
                "upper_cap": upper_cap,
            }
        )

    return cleaned, pd.DataFrame(report).set_index("column")


measurements = pd.DataFrame(
    {
        "C1": [-21.57, 1.24, -0.84, 32.35, 0.82, -3.11, 0.46, -18.68, 0.04, 30.87],
        "C2": [2.49, 2.27, 0.25, -2.33, -2.91, -3.61, 0.58, -2.59, -3.99, 1.54],
        "C3": [1.14, 4.76, 14.23, -2.65, -3.53, -0.03, 17.5, -0.21, 2.96, 2.31],
    }
)

clean_measurements, outlier_report = cap_outliers_iqr(measurements)
outlier_report


,outlier_count,lower_fence,upper_fence,lower_cap,upper_cap
column,,,,,
C1,4,-8.05875,6.65125,-3.11,1.24
C2,0,-9.02500,7.49500,-3.99,2.49
C3,2,-6.87750,11.02250,-3.53,4.76


In [25]:
clean_measurements


,C1,C2,C3
0,-3.11,2.49,1.14
1,1.24,2.27,4.76
2,-0.84,0.25,4.76
3,1.24,-2.33,-2.65
4,0.82,-2.91,-3.53
5,-3.11,-3.61,-0.03
6,0.46,0.58,4.76
7,-3.11,-2.59,-0.21
8,0.04,-3.99,2.96
9,1.24,1.54,2.31


## 3. Classement selon le rapport prix/score

Les scores sont agrégés par marque, puis joints aux prix. Un rapport prix/score plus faible correspond ici à un meilleur classement. Les marques sans score valide ou avec un score non positif sont exclues.


In [26]:
def rank_products(scores: pd.DataFrame, prices: pd.DataFrame) -> pd.DataFrame:
    """Classe les produits par rapport prix/score moyen croissant."""
    required_scores = {"BrandName", "Score"}
    required_prices = {"BrandName", "Price"}
    if missing := required_scores - set(scores.columns):
        raise KeyError(f"Colonnes absentes de scores : {sorted(missing)}")
    if missing := required_prices - set(prices.columns):
        raise KeyError(f"Colonnes absentes de prices : {sorted(missing)}")

    average_scores = (
        scores.groupby("BrandName", as_index=False, dropna=False)["Score"]
        .mean()
        .rename(columns={"Score": "AvgScore"})
    )
    ranked = average_scores.merge(
        prices[["BrandName", "Price"]], on="BrandName", how="inner", validate="one_to_one"
    )
    ranked = ranked.loc[ranked["AvgScore"].gt(0) & ranked["Price"].notna()].copy()
    ranked["PricePerScore"] = ranked["Price"] / ranked["AvgScore"]

    return ranked.sort_values("PricePerScore", ignore_index=True)


scores = pd.DataFrame(
    {
        "BrandName": ["Lowell", "Lowell", "Purewash", "Wilson", "Purewash", "Wilson"],
        "Score": [0.743, 0.741, 0.784, np.nan, 0.968, 0.739],
    }
)
prices = pd.DataFrame(
    {"BrandName": ["Lowell", "Purewash", "Wilson"], "Price": [135.75, 122.61, 113.67]}
)

rank_products(scores, prices)


,BrandName,AvgScore,Price,PricePerScore
0,Purewash,0.876,122.61,139.965753
1,Wilson,0.739,113.67,153.815968
2,Lowell,0.742,135.75,182.951482


## 4. Tokenisation simple

Cette tokenisation retire la ponctuation ASCII, normalise la casse et associe chaque mot à un indice déterministe. Pour du traitement automatique du langage en production, une bibliothèque spécialisée sera généralement préférable.


In [27]:
def tokenize_sentence(sentence: str) -> tuple[list[int], dict[str, int]]:
    """Retourne les indices des mots et le vocabulaire trié associé."""
    if not isinstance(sentence, str):
        raise TypeError("sentence doit être une chaîne de caractères")

    translation = str.maketrans("", "", string.punctuation)
    words = sentence.translate(translation).casefold().split()
    vocabulary = {word: index for index, word in enumerate(sorted(set(words)))}
    tokens = [vocabulary[word] for word in words]
    return tokens, vocabulary


tokens, vocabulary = tokenize_sentence(
    "They know her, Jill, but she does not know them."
)
tokens, vocabulary


([8, 4, 2, 3, 0, 6, 1, 5, 4, 7],
 {'but': 0,
  'does': 1,
  'her': 2,
  'jill': 3,
  'know': 4,
  'not': 5,
  'she': 6,
  'them': 7,
  'they': 8})

## 5. Standardisation ligne par ligne

Chaque ligne est centrée et réduite indépendamment. La stratégie appliquée aux lignes constantes est explicite : elles peuvent produire des `NaN`, des valeurs infinies, des zéros, ou déclencher une erreur.


In [28]:
def standardize_rows(
    values,
    *,
    constant_policy: str = "nan",
) -> np.ndarray:
    """Calcule le z-score de chaque ligne d'une matrice bidimensionnelle."""
    array = np.asarray(values, dtype=float)
    if array.ndim != 2:
        raise ValueError("values doit être une matrice bidimensionnelle")
    if constant_policy not in {"nan", "inf", "zero", "raise"}:
        raise ValueError("constant_policy doit valoir 'nan', 'inf', 'zero' ou 'raise'")

    means = array.mean(axis=1, keepdims=True)
    standard_deviations = array.std(axis=1, keepdims=True)
    constant_rows = standard_deviations[:, 0] == 0
    if constant_policy == "raise" and constant_rows.any():
        indices = np.flatnonzero(constant_rows).tolist()
        raise ValueError(f"Lignes constantes détectées : {indices}")

    safe_deviations = np.where(standard_deviations == 0, 1.0, standard_deviations)
    standardized = (array - means) / safe_deviations
    replacement = {"nan": np.nan, "inf": np.inf, "zero": 0.0}.get(constant_policy)
    if replacement is not None:
        standardized[constant_rows] = replacement
    return standardized


matrix = np.array([[1, 2, 3], [4, 4, 4], [10, 20, 30]])
standardize_rows(matrix, constant_policy="nan")


array([[-1.22474487,  0.        ,  1.22474487],
       [        nan,         nan,         nan],
       [-1.22474487,  0.        ,  1.22474487]])

## 6. Construction d'une matrice de caractéristiques

Les colonnes numériques sont conservées et l'année d'obtention du diplôme est encodée en *one-hot*. La fonction retourne un `DataFrame` afin de préserver les noms de caractéristiques ; `to_numpy()` peut ensuite être utilisé par les bibliothèques qui exigent une matrice NumPy.


In [29]:
def build_feature_matrix(students: pd.DataFrame) -> pd.DataFrame:
    """Combine les scores et l'encodage one-hot de l'année de diplôme."""
    score_columns = ["MeanScore", "MedianScore"]
    required = {"GradYear", *score_columns}
    if missing := required - set(students.columns):
        raise KeyError(f"Colonnes absentes : {sorted(missing)}")

    scores = students[score_columns].reset_index(drop=True)
    years = pd.get_dummies(
        students["GradYear"], prefix="GradYear", dtype=float
    ).reset_index(drop=True)
    return pd.concat([scores, years], axis=1)


students = pd.DataFrame(
    {
        "FirstName": ["Jack", "Sam", "Jenn", "Beth"],
        "LastName": ["Doe", "Williams", "Smith", "Parker"],
        "GradYear": [2021, 2022, 2023, 2022],
        "MeanScore": [90.6, 88.7, 74.2, 89.5],
        "MedianScore": [90.0, 88.5, 75.5, 89.1],
    }
)

feature_matrix = build_feature_matrix(students)
feature_matrix


,MeanScore,MedianScore,GradYear_2021,GradYear_2022,GradYear_2023
0,90.6,90.0,1.0,0.0,0.0
1,88.7,88.5,0.0,1.0,0.0
2,74.2,75.5,0.0,0.0,1.0
3,89.5,89.1,0.0,1.0,0.0


## 7. Prétraitement de variables catégorielles

La casse et les espaces sont normalisés avant l'encodage. Les valeurs manquantes reçoivent une catégorie dédiée plutôt qu'une valeur métier arbitraire. `handle_unknown="ignore"` permet de transformer ultérieurement des données contenant de nouvelles catégories.


In [30]:
def clean_categorical_columns(
    dataframe: pd.DataFrame,
    columns: Iterable[str],
    *,
    missing_label: str = "missing",
) -> pd.DataFrame:
    """Normalise des colonnes textuelles sans modifier le DataFrame source."""
    columns = list(columns)
    missing = sorted(set(columns) - set(dataframe.columns))
    if missing:
        raise KeyError(f"Colonnes absentes : {missing}")

    cleaned = dataframe[columns].copy()
    for column in columns:
        cleaned[column] = (
            cleaned[column]
            .astype("string")
            .str.strip()
            .str.casefold()
            .fillna(missing_label)
        )
    return cleaned


def one_hot_encode(
    dataframe: pd.DataFrame,
    columns: Iterable[str],
) -> tuple[pd.DataFrame, OneHotEncoder]:
    """Nettoie puis encode des variables catégorielles avec scikit-learn."""
    columns = list(columns)
    cleaned = clean_categorical_columns(dataframe, columns)
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=float)
    values = encoder.fit_transform(cleaned)
    encoded = pd.DataFrame(
        values,
        columns=encoder.get_feature_names_out(columns),
        index=dataframe.index,
    )
    return encoded, encoder


market_data = pd.DataFrame(
    {
        "transactions": [243243, 578123, 351780, 34098, 34098],
        "mood": ["crash", "stable", "good", None, "Stable"],
        "restrictions": ["banned", "Regulated", "free", "regulated", "free"],
        "integrations": ["noIntegration", "PayPal", "nointegration", "yandex", "Paypal"],
        "media": [0.6, 0.7, 0.8, 0.9, 0.9],
        "price": [9008, 16065, 17076, 19341, 4789],
    }
)

categorical_columns = ["mood", "restrictions", "integrations"]
encoded_market_data, fitted_encoder = one_hot_encode(
    market_data, categorical_columns
)
encoded_market_data


,mood_crash,mood_good,mood_missing,mood_stable,restrictions_banned,restrictions_free,restrictions_regulated,integrations_nointegration,integrations_paypal,integrations_yandex
0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
2,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
3,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
4,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0


## 8. Tests de conformité aux résultats de référence

Cette cellule reprend exactement les entrées et sorties transcrites depuis les captures. Les adaptateurs conservent les fonctions enrichies tout en exposant les signatures attendues par les exercices. Pour les nombres flottants, l'écart affiché est l'erreur absolue maximale.


In [31]:
def anonymize_data(dataframe: pd.DataFrame) -> pd.DataFrame:
    encoded, _ = encode_categories(dataframe, ["CustomerName", "Order"])
    return encoded


def rank_washing_machines(scores_df: pd.DataFrame, prices_df: pd.DataFrame) -> pd.DataFrame:
    return rank_products(scores_df, prices_df)[["BrandName", "Price", "AvgScore"]]


_tokenize_sentence_with_vocabulary = tokenize_sentence


def tokenize_sentence(sentence: str) -> list[int]:
    tokens, _ = _tokenize_sentence_with_vocabulary(sentence)
    return tokens


def standardize_data(values) -> np.ndarray:
    return standardize_rows(values, constant_policy="inf")


def df2matrix(dataframe: pd.DataFrame) -> np.ndarray:
    required = {"GradYear", "MeanScore", "MedianScore"}
    if missing := required - set(dataframe.columns):
        raise KeyError(f"Colonnes absentes : {sorted(missing)}")
    scores = dataframe[["MeanScore", "MedianScore"]].to_numpy(dtype=float)
    years = pd.get_dummies(dataframe["GradYear"], dtype=float).reindex(
        columns=[2021, 2022, 2023], fill_value=0.0
    )
    return np.hstack([scores, years.to_numpy()])


def preprocess(dataframe: pd.DataFrame) -> pd.DataFrame:
    columns = ["mood", "restrictions", "integrations"]
    cleaned = clean_categorical_columns(dataframe, columns, missing_label="stable")
    return pd.get_dummies(cleaned, columns=columns, dtype=int)


test_results = []

# Encodage des clients et commandes
orders_test = pd.DataFrame({
    "CustomerName": ["John Smith", "Tom Smith", "Jane Doe", "Alex Williams", "John Smith", "Jane Doe"],
    "Order": ["Laptop", "Router", "TV", "Laptop", "TV", "Ethernet Cable"],
})
orders_expected = pd.DataFrame(
    {"CustomerName": [3, 4, 2, 1, 3, 2], "Order": [2, 3, 4, 2, 4, 1]}, dtype="Int64"
)
orders_ok = anonymize_data(orders_test).equals(orders_expected)
test_results.append({"test": "anonymize_data", "max_abs_error": 0.0 if orders_ok else np.nan, "status": "OK" if orders_ok else "ÉCHEC"})

# Classement des machines
scores_test = pd.DataFrame({
    "BrandName": ["Lowell", "Lowell", "Purewash", "Wilson", "Purewash", "Purewash", "Lowell", "Wilson", "Lowell", "Wilson", "Purewash", "Wilson"],
    "Score": [0.743, 0.741, 0.784, np.nan, 0.968, 0.840, np.nan, np.nan, 0.451, 0.739, 0.539, 0.485],
})
prices_test = pd.DataFrame({"BrandName": ["Lowell", "Purewash", "Wilson"], "Price": [135.75, 122.61, 113.67]})
ranking_expected = pd.DataFrame({"BrandName": ["Purewash", "Wilson", "Lowell"], "Price": [122.61, 113.67, 135.75], "AvgScore": [0.78275, 0.612, 0.645]})
ranking_actual = rank_washing_machines(scores_test, prices_test)
ranking_error = float(np.max(np.abs(ranking_actual[["Price", "AvgScore"]].to_numpy() - ranking_expected[["Price", "AvgScore"]].to_numpy())))
ranking_ok = ranking_actual["BrandName"].tolist() == ranking_expected["BrandName"].tolist() and ranking_error < 1e-12
test_results.append({"test": "rank_washing_machines", "max_abs_error": ranking_error, "status": "OK" if ranking_ok else "ÉCHEC"})

# Tokenisation : nombre maximal de positions différentes sur les trois exemples
token_cases = [
    ("they know her , jill . but she does not know them .", [8, 4, 2, 3, 0, 6, 1, 5, 4, 7]),
    ("we came , we saw , we conquered .", [3, 0, 3, 2, 3, 1]),
    ("square : a shape that is a rectangle", [4, 0, 3, 5, 1, 0, 2]),
]
token_errors = [sum(a != b for a, b in zip(tokenize_sentence(text), expected)) for text, expected in token_cases]
test_results.append({"test": "tokenize_sentence (3 cas)", "max_abs_error": max(token_errors), "status": "OK" if token_errors == [0, 0, 0] else "ÉCHEC"})

# Standardisation : comparaison des valeurs finies et de la position des infinis
standardization_input = np.array([[10.3, 0.5, -1.2], [1.0, 1.0, 1.0], [-5.4, 10.0, 1.0]])
standardization_expected = np.array([[1.40089141, -0.53273335, -0.86815806], [np.inf, np.inf, np.inf], [-1.15036775, 1.28756757, -0.13719982]])
standardization_actual = standardize_data(standardization_input)
finite_mask = np.isfinite(standardization_expected)
standardization_error = float(np.max(np.abs(standardization_actual[finite_mask] - standardization_expected[finite_mask])))
standardization_ok = standardization_error < 1e-7 and np.array_equal(np.isinf(standardization_actual), np.isinf(standardization_expected))
test_results.append({"test": "standardize_data", "max_abs_error": standardization_error, "status": "OK" if standardization_ok else "ÉCHEC"})

# Matrice de caractéristiques
matrix_expected = np.array([[90.6, 90.0, 1.0, 0.0, 0.0], [88.7, 88.5, 0.0, 1.0, 0.0], [74.2, 75.5, 0.0, 0.0, 1.0], [89.5, 89.1, 0.0, 1.0, 0.0]])
matrix_error = float(np.max(np.abs(df2matrix(students) - matrix_expected)))
test_results.append({"test": "df2matrix", "max_abs_error": matrix_error, "status": "OK" if matrix_error < 1e-12 else "ÉCHEC"})

# Prétraitement catégoriel
preprocess_expected = np.array([[1, 0, 0, 1, 0, 0, 1, 0, 0], [0, 0, 1, 0, 0, 1, 0, 1, 0], [0, 1, 0, 0, 1, 0, 1, 0, 0], [0, 0, 1, 0, 0, 1, 0, 0, 1], [0, 0, 1, 0, 1, 0, 0, 1, 0]])
preprocess_error = float(np.max(np.abs(preprocess(market_data).to_numpy() - preprocess_expected)))
test_results.append({"test": "preprocess", "max_abs_error": preprocess_error, "status": "OK" if preprocess_error == 0 else "ÉCHEC"})

conformance_report = pd.DataFrame(test_results).set_index("test")
conformance_report


,max_abs_error,status
test,,
anonymize_data,0.000000e+00,OK
rank_washing_machines,1.110223e-16,OK
tokenize_sentence (3 cas),0.000000e+00,OK
standardize_data,4.885176e-09,OK
df2matrix,0.000000e+00,OK
preprocess,0.000000e+00,OK


In [32]:
assert (conformance_report["status"] == "OK").all(), conformance_report
print("Les six familles de résultats correspondent aux captures de référence.")


Les six familles de résultats correspondent aux captures de référence.


## Bonnes pratiques pour la suite

- Ajuster les transformations uniquement sur le jeu d'entraînement afin d'éviter les fuites de données.
- Conserver les encodeurs et leurs paramètres avec le modèle déployé.
- Ajouter des tests unitaires pour les valeurs manquantes, les catégories inconnues et les entrées invalides.
- Versionner les dépendances dans `requirements.txt` ou `pyproject.toml`.
- Ne pas publier de données personnelles ni de tables de correspondance sensibles.
